In [ ]:
!pip install roboflow ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 96.4 MB/s eta 0:00:00


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="ucL6xwgmpE7w8Bl4jzBx")
project = rf.workspace("xaviervape-old").project("solid-waste-ajntx")
version = project.version(1)
dataset1 = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Solid-Waste-1 in yolov11:: 100%|██████████| 278/278 [00:00<00:00, 1460.00it/s]


In [ ]:
import os, yaml

label_dir = f"{dataset1.location}/train/labels"
sample = os.listdir(label_dir)[0]
with open(f"{label_dir}/{sample}", 'r') as f:
    line = f.readline().strip()

values = line.split()
print(f"Values per line: {len(values)}")
print("✅ Segmentation confirmed!" if len(values) > 5 else "❌ Detection only")

with open(f"{dataset1.location}/data.yaml", 'r') as f:
    data = yaml.safe_load(f)
print(f"\nClasses: {data['names']}")
print(f"Total classes: {data['nc']}")

Values per line: 1063
✅ Segmentation confirmed!

Classes: ['Clothing', 'General trash', 'Glass', 'Metal', 'Paper', 'Paper pack', 'Plastic', 'Plastic bag', 'Styrofoam']
Total classes: 9


In [ ]:
import os, yaml

dataset_path = dataset1.location

ORIGINAL_CLASSES = ['Clothing', 'General trash', 'Glass', 'Metal',
                    'Paper', 'Paper pack', 'Plastic', 'Plastic bag', 'Styrofoam']

CLASS_MAPPING = {
    'Glass':       'glass',
    'Metal':       'metal_scrap',
    'Paper':       'paper',
    'Paper pack':  'cardboard',
    'Plastic':     'plastic',
    'Plastic bag': 'plastic',
}

REMOVE_CLASSES = {'Clothing', 'General trash', 'Styrofoam'}
NEW_CLASSES = ['glass', 'metal_scrap', 'paper', 'cardboard', 'plastic']

def clean_labels(split):
    label_dir = os.path.join(dataset_path, split, 'labels')
    image_dir = os.path.join(dataset_path, split, 'images')
    kept = 0
    removed = 0

    for label_file in os.listdir(label_dir):
        if not label_file.endswith('.txt'):
            continue

        label_path = os.path.join(label_dir, label_file)
        with open(label_path, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            old_cls_id = int(parts[0])
            old_cls_name = ORIGINAL_CLASSES[old_cls_id]

            if old_cls_name in REMOVE_CLASSES:
                continue

            new_cls_name = CLASS_MAPPING[old_cls_name]
            new_cls_id = NEW_CLASSES.index(new_cls_name)
            parts[0] = str(new_cls_id)
            new_lines.append(' '.join(parts) + '\n')

        if not new_lines:
            os.remove(label_path)
            for ext in ['.jpg', '.jpeg', '.png']:
                img = os.path.join(image_dir, label_file.replace('.txt', ext))
                if os.path.exists(img):
                    os.remove(img)
            removed += 1
        else:
            with open(label_path, 'w') as f:
                f.writelines(new_lines)
            kept += 1

    print(f"{split}: kept {kept}, removed {removed}")

for split in ['train', 'valid', 'test']:
    clean_labels(split)

# Update data.yaml
new_yaml = {
    'path': dataset_path,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    len(NEW_CLASSES),
    'names': NEW_CLASSES
}
with open(os.path.join(dataset_path, 'data.yaml'), 'w') as f:
    yaml.dump(new_yaml, f)

print(f"\nFinal classes: {NEW_CLASSES}")
print("Dataset 1 ready!")

train: kept 113, removed 0
valid: kept 10, removed 0
test: kept 10, removed 0

Final classes: ['glass', 'metal_scrap', 'paper', 'cardboard', 'plastic']
Dataset 1 ready!


In [ ]:
import os, yaml

with open(f"{dataset1.location}/data.yaml", 'r') as f:
    data = yaml.safe_load(f)

print(f"Classes: {data['names']}")
print(f"Total classes: {data['nc']}")

for split in ['train', 'valid', 'test']:
    count = len(os.listdir(f"{dataset1.location}/{split}/images"))
    print(f"{split}: {count} images")

Classes: ['glass', 'metal_scrap', 'paper', 'cardboard', 'plastic']
Total classes: 5
train: 113 images
valid: 10 images
test: 10 images


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="ucL6xwgmpE7w8Bl4jzBx")

project = rf.workspace("ipcv-bvagm").project("ipcv-heu1q")
version = project.version(1)
dataset_ewaste = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to IPCV-1 in yolov11:: 100%|██████████| 1021/1021 [00:00<00:00, 7183.14it/s]


In [ ]:
import os, yaml

label_dir = f"{dataset_ewaste.location}/train/labels"
sample = os.listdir(label_dir)[0]
with open(f"{label_dir}/{sample}", 'r') as f:
    line = f.readline().strip()

values = line.split()
print(f"Values per line: {len(values)}")
print("✅ Segmentation!" if len(values) > 5 else "❌ Detection only")

with open(f"{dataset_ewaste.location}/data.yaml", 'r') as f:
    data = yaml.safe_load(f)
print(f"Classes: {data['names']}")
print(f"Total images: {len(os.listdir(f'{dataset_ewaste.location}/train/images'))}")

Values per line: 23
✅ Segmentation!
Classes: ['E-waste-Laptops', 'E-waste-Monitors', 'Mixed-Recyclables-Cardboard', 'Mixed-Recyclables-Detergent-boxes', 'Organic-Waste-Bread', 'Organic-Waste-Pasta']
Total images: 357


In [ ]:
import os, yaml

dataset_ewaste_path = dataset_ewaste.location

ORIGINAL_CLASSES_EW = [
    'E-waste-Laptops',
    'E-waste-Monitors',
    'Mixed-Recyclables-Cardboard',
    'Mixed-Recyclables-Detergent-boxes',
    'Organic-Waste-Bread',
    'Organic-Waste-Pasta'
]

CLASS_MAPPING_EW = {
    'E-waste-Laptops':                  'ewaste',
    'E-waste-Monitors':                 'ewaste',
    'Mixed-Recyclables-Cardboard':      'cardboard',
    'Mixed-Recyclables-Detergent-boxes':'plastic',
}

REMOVE_CLASSES_EW = {'Organic-Waste-Bread', 'Organic-Waste-Pasta'}

# Final combined classes (Dataset 1 + Dataset 2)
FINAL_CLASSES = ['glass', 'metal_scrap', 'paper', 'cardboard', 'plastic', 'ewaste']

def clean_ewaste_labels(split):
    label_dir = os.path.join(dataset_ewaste_path, split, 'labels')
    image_dir = os.path.join(dataset_ewaste_path, split, 'images')
    kept = 0
    removed = 0

    for label_file in os.listdir(label_dir):
        if not label_file.endswith('.txt'):
            continue

        label_path = os.path.join(label_dir, label_file)
        with open(label_path, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            old_cls_id = int(parts[0])
            old_cls_name = ORIGINAL_CLASSES_EW[old_cls_id]

            if old_cls_name in REMOVE_CLASSES_EW:
                continue

            new_cls_name = CLASS_MAPPING_EW[old_cls_name]
            new_cls_id = FINAL_CLASSES.index(new_cls_name)
            parts[0] = str(new_cls_id)
            new_lines.append(' '.join(parts) + '\n')

        if not new_lines:
            os.remove(label_path)
            for ext in ['.jpg', '.jpeg', '.png']:
                img = os.path.join(image_dir, label_file.replace('.txt', ext))
                if os.path.exists(img):
                    os.remove(img)
            removed += 1
        else:
            with open(label_path, 'w') as f:
                f.writelines(new_lines)
            kept += 1

    print(f"{split}: kept {kept}, removed {removed}")

for split in ['train', 'valid', 'test']:
    clean_ewaste_labels(split)

print(f"\nE-waste dataset cleaned!")

train: kept 197, removed 160
valid: kept 58, removed 43
test: kept 27, removed 23

E-waste dataset cleaned!


In [ ]:
import zipfile, os

zip_path = "/content/Kabadiai.yolov11.zip"
extract_path = "/content/newspaper_seg"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted!")

# Check structure
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = '  ' * (level + 1)
        for file in files[:2]:
            print(f"{subindent}{file}")

Extracted!
newspaper_seg/
  README.roboflow.txt
  README.dataset.txt
  valid/
    labels/
    images/
  test/
    labels/
    images/
  train/
    labels/
    images/


In [ ]:
import yaml, os

with open(f"{extract_path}/data.yaml", 'r') as f:
    data = yaml.safe_load(f)
print(f"Classes: {data['names']}")
print(f"Total classes: {data['nc']}")

# Check label format
label_dir = f"{extract_path}/train/labels"
sample = os.listdir(label_dir)[0]
with open(f"{label_dir}/{sample}", 'r') as f:
    line = f.readline().strip()
values = line.split()
print(f"\nValues per line: {len(values)}")
print("✅ Segmentation!" if len(values) > 5 else "❌ Detection only")

# Count images
for split in ['train', 'valid', 'test']:
    count = len(os.listdir(f"{extract_path}/{split}/images"))
    print(f"{split}: {count} images")

Classes: ['newspaper']
Total classes: 1

Values per line: 815
✅ Segmentation!
train: 347 images
valid: 99 images
test: 50 images


In [ ]:
import os, shutil

MERGED_PATH = "/content/kabadiai_merged"
FINAL_CLASSES = ['glass', 'metal_scrap', 'paper', 'cardboard', 'plastic', 'ewaste']
paper_idx = FINAL_CLASSES.index('paper')

for split in ['train', 'valid', 'test']:
    os.makedirs(f"{MERGED_PATH}/{split}/images", exist_ok=True)
    os.makedirs(f"{MERGED_PATH}/{split}/labels", exist_ok=True)

def merge_split(src_path, split, prefix, remap_to_idx=None):
    img_src = os.path.join(src_path, split, 'images')
    lbl_src = os.path.join(src_path, split, 'labels')
    img_dst = f"{MERGED_PATH}/{split}/images"
    lbl_dst = f"{MERGED_PATH}/{split}/labels"

    count = 0
    for fname in os.listdir(img_src):
        name, ext = os.path.splitext(fname)
        new_name = f"{prefix}_{name}"
        shutil.copy(f"{img_src}/{fname}", f"{img_dst}/{new_name}{ext}")

        lbl_file = f"{lbl_src}/{name}.txt"
        if os.path.exists(lbl_file):
            if remap_to_idx is not None:
                # force every box/polygon in this file to one class id
                with open(lbl_file, 'r') as f:
                    lines = f.readlines()
                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    parts[0] = str(remap_to_idx)
                    new_lines.append(' '.join(parts) + '\n')
                with open(f"{lbl_dst}/{new_name}.txt", 'w') as f:
                    f.writelines(new_lines)
            else:
                shutil.copy(lbl_file, f"{lbl_dst}/{new_name}.txt")
            count += 1
    print(f"  {split}: copied {count} files from {prefix}")

print("Merging Dataset 1 (solid waste)...")
for split in ['train', 'valid', 'test']:
    merge_split(dataset1.location, split, "sw")

print("Merging Dataset 2 (ewaste)...")
for split in ['train', 'valid', 'test']:
    merge_split(dataset_ewaste.location, split, "ew")

print("Merging newspaper (remapped to 'paper')...")
for split in ['train', 'valid', 'test']:
    merge_split(extract_path, split, "np", remap_to_idx=paper_idx)

print("\n✅ Merge complete!")
for split in ['train', 'valid', 'test']:
    count = len(os.listdir(f"{MERGED_PATH}/{split}/images"))
    print(f"  {split}: {count} total images")

Merging Dataset 1 (solid waste)...
  train: copied 113 files from sw
  valid: copied 10 files from sw
  test: copied 10 files from sw
Merging Dataset 2 (ewaste)...
  train: copied 197 files from ew
  valid: copied 58 files from ew
  test: copied 27 files from ew
Merging newspaper (remapped to 'paper')...
  train: copied 347 files from np
  valid: copied 99 files from np
  test: copied 50 files from np

✅ Merge complete!
  train: 657 total images
  valid: 167 total images
  test: 87 total images


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.make_archive('/content/drive/MyDrive/KabadiAI/kabadiai_merged_backup', 'zip', MERGED_PATH)
print("✅ Backed up to Drive!")

Mounted at /content/drive
✅ Backed up to Drive!


In [ ]:
import cv2, os
import numpy as np

def augment_newspaper_split(split, num_augs_per_image=2):
    img_dir = f"{MERGED_PATH}/{split}/images"
    lbl_dir = f"{MERGED_PATH}/{split}/labels"
    np_images = [f for f in os.listdir(img_dir) if f.startswith('np_')]

    created = 0
    for fname in np_images:
        name, ext = os.path.splitext(fname)
        img_path = f"{img_dir}/{fname}"
        lbl_path = f"{lbl_dir}/{name}.txt"
        if not os.path.exists(lbl_path):
            continue

        img = cv2.imread(img_path)
        h, w = img.shape[:2]

        with open(lbl_path, 'r') as f:
            lines = f.readlines()

        for aug_i in range(num_augs_per_image):
            scale = np.random.uniform(0.4, 0.7)
            new_w, new_h = int(w * scale), int(h * scale)
            resized = cv2.resize(img, (new_w, new_h))

            canvas = np.full((h, w, 3), 255, dtype=np.uint8)
            off_x = np.random.randint(0, w - new_w + 1)
            off_y = np.random.randint(0, h - new_h + 1)
            canvas[off_y:off_y+new_h, off_x:off_x+new_w] = resized

            new_lines = []
            for line in lines:
                parts = line.strip().split()
                cls_id = parts[0]
                coords = list(map(float, parts[1:]))
                new_coords = []
                for j in range(0, len(coords), 2):
                    x, y = coords[j], coords[j+1]
                    px, py = x * w, y * h
                    px, py = px * scale + off_x, py * scale + off_y
                    new_coords.append(px / w)
                    new_coords.append(py / h)
                new_lines.append(cls_id + ' ' + ' '.join(f"{c:.6f}" for c in new_coords) + '\n')

            new_name = f"{name}_aug{aug_i}"
            cv2.imwrite(f"{img_dir}/{new_name}{ext}", canvas)
            with open(f"{lbl_dir}/{new_name}.txt", 'w') as f:
                f.writelines(new_lines)
            created += 1

    print(f"  {split}: created {created} zoomed-out newspaper variants")

for split in ['train', 'valid', 'test']:
    augment_newspaper_split(split, num_augs_per_image=2 if split=='train' else 1)

print("\n✅ Augmentation complete! New totals:")
for split in ['train', 'valid', 'test']:
    count = len(os.listdir(f"{MERGED_PATH}/{split}/images"))
    print(f"  {split}: {count} total images")

  train: created 694 zoomed-out newspaper variants
  valid: created 99 zoomed-out newspaper variants
  test: created 50 zoomed-out newspaper variants

✅ Augmentation complete! New totals:
  train: 1351 total images
  valid: 266 total images
  test: 137 total images


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.make_archive('/content/drive/MyDrive/KabadiAI/kabadiai_merged_backup', 'zip', MERGED_PATH)
print("✅ Backed up to Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Backed up to Drive!


In [ ]:
import yaml

FINAL_CLASSES = ['glass', 'metal_scrap', 'paper', 'cardboard', 'plastic', 'ewaste']

data_yaml = {
    'path': MERGED_PATH,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(FINAL_CLASSES),
    'names': FINAL_CLASSES
}

with open(f"{MERGED_PATH}/data.yaml", 'w') as f:
    yaml.dump(data_yaml, f)

print("✅ data.yaml written")
print(data_yaml)

✅ data.yaml written
{'path': '/content/kabadiai_merged', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 6, 'names': ['glass', 'metal_scrap', 'paper', 'cardboard', 'plastic', 'ewaste']}


In [ ]:
from collections import Counter

class_counts = Counter()
empty_files = 0
bad_index = 0

for split in ['train', 'valid', 'test']:
    lbl_dir = f"{MERGED_PATH}/{split}/labels"
    for fname in os.listdir(lbl_dir):
        with open(f"{lbl_dir}/{fname}", 'r') as f:
            lines = f.readlines()
        if not lines:
            empty_files += 1
            continue
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            idx = int(parts[0])
            if idx < 0 or idx >= len(FINAL_CLASSES):
                bad_index += 1
            else:
                class_counts[FINAL_CLASSES[idx]] += 1

print("Class distribution (instance count):")
for cls, count in class_counts.most_common():
    print(f"  {cls:12} {count}")
print(f"\nEmpty label files: {empty_files}")
print(f"Bad class indices: {bad_index}")

Class distribution (instance count):
  paper        1584
  plastic      656
  ewaste       300
  glass        272
  cardboard    181
  metal_scrap  72

Empty label files: 175
Bad class indices: 0


In [ ]:
empty_by_prefix = Counter()
for split in ['train', 'valid', 'test']:
    lbl_dir = f"{MERGED_PATH}/{split}/labels"
    for fname in os.listdir(lbl_dir):
        with open(f"{lbl_dir}/{fname}", 'r') as f:
            content = f.read().strip()
        if not content:
            prefix = fname.split('_')[0]  # sw / ew / np
            empty_by_prefix[prefix] += 1

print(empty_by_prefix)

Counter({'np': 175})


In [ ]:
# Inspect a few of the empty np label files and find their image counterpart
lbl_dir = f"{MERGED_PATH}/train/labels"
img_dir = f"{MERGED_PATH}/train/images"

empty_np_files = []
for fname in os.listdir(lbl_dir):
    if fname.startswith('np_'):
        with open(f"{lbl_dir}/{fname}", 'r') as f:
            if not f.read().strip():
                empty_np_files.append(fname)

print(f"Total empty np label files: {len(empty_np_files)}")
print("Sample names:", empty_np_files[:10])

# Check if these are originals or _aug variants
orig_empty = [f for f in empty_np_files if '_aug' not in f]
aug_empty = [f for f in empty_np_files if '_aug' in f]
print(f"\nOriginal (non-aug) empty: {len(orig_empty)}")
print(f"Augmented (_aug) empty: {len(aug_empty)}")

Total empty np label files: 153
Sample names: ['np_111binder_Page_178_jpg.rf.fFvvuOXotq0N3h6UQZXr.txt', 'np_111binder_Page_199_jpg.rf.jE4CId4dck89rVUAnOA4_aug1.txt', 'np_111binder_Page_214_jpg.rf.CcVv6rZRyM18KasXHfmk_aug0.txt', 'np_111binder_Page_161_jpg.rf.PBoInPQHX3h9ilrALpG4.txt', 'np_111binder_Page_038_jpg.rf.MVNKwuFOtXA5HIw0ZNTr_aug1.txt', 'np_1binder_Page_074_jpg.rf.5PP3bTk4he04PnGpTgHa.txt', 'np_1binder_Page_030_jpg.rf.2VHJlXAyPG0kN2EQAHSh_aug1.txt', 'np_111binder_Page_193_jpg.rf.GQzH9gCBgmoGqRqyueG2_aug0.txt', 'np_DB-Delhi-02-July-2023_page-0008_jpg.rf.vn76gmPlFOgK8j1CKo95_aug0.txt', 'np_111binder_Page_214_jpg.rf.CcVv6rZRyM18KasXHfmk.txt']

Original (non-aug) empty: 51
Augmented (_aug) empty: 102


In [ ]:
print(f"Mismatch check: found {len(empty_np_files)} in train split only")
# Earlier count of 175 was across train+valid+test combined
for split in ['valid', 'test']:
    lbl_dir = f"{MERGED_PATH}/{split}/labels"
    cnt = sum(1 for f in os.listdir(lbl_dir) if f.startswith('np_') and not open(f"{lbl_dir}/{f}").read().strip())
    print(f"{split}: {cnt} empty np files")

Mismatch check: found 153 in train split only
valid: 16 empty np files
test: 6 empty np files


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m-seg.pt')  # pretrained, fine-tune on our merged dataset

results = model.train(
    data=f"{MERGED_PATH}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,        # early stop if val mAP plateaus
    project='/content/drive/MyDrive/KabadiAI/runs',  # save directly to Drive this time
    name='kabadiai_v2',
    save=True,
    save_period=10,      # checkpoint every 10 epochs, extra insurance against runtime resets
)

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/kabadiai_merged/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kabadiai_v2-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_ma

In [ ]:
final_model_dir = "/content/drive/MyDrive/KabadiAI/models"
os.makedirs(final_model_dir, exist_ok=True)

shutil.copy(best_pt_path, f"{final_model_dir}/kabadiai_best.pt")
print(f"✅ Saved to {final_model_dir}/kabadiai_best.pt")

NameError: name 'best_pt_path' is not defined

In [ ]:
import os, shutil

runs_dir = "/content/drive/MyDrive/KabadiAI/runs"
run_folders = sorted(
    [f for f in os.listdir(runs_dir) if f.startswith("kabadiai_v2")],
    key=lambda x: os.path.getmtime(os.path.join(runs_dir, x))
)
print("Found run folders:", run_folders)

latest_run = run_folders[-1]
print("Latest run folder:", latest_run)

best_pt_path = f"{runs_dir}/{latest_run}/weights/best.pt"
print("best.pt path:", best_pt_path)
print("best.pt exists:", os.path.exists(best_pt_path))

Found run folders: ['kabadiai_v2', 'kabadiai_v2-2']
Latest run folder: kabadiai_v2-2
best.pt path: /content/drive/MyDrive/KabadiAI/runs/kabadiai_v2-2/weights/best.pt
best.pt exists: True


In [ ]:
final_model_dir = "/content/drive/MyDrive/KabadiAI/models"
os.makedirs(final_model_dir, exist_ok=True)

shutil.copy(best_pt_path, f"{final_model_dir}/kabadiai_best.pt")
print(f"✅ Saved to {final_model_dir}/kabadiai_best.pt")

✅ Saved to /content/drive/MyDrive/KabadiAI/models/kabadiai_best.pt


In [ ]:
weights_dir = f"{runs_dir}/{latest_run}/weights"
print("Weights folder contents:", os.listdir(weights_dir))

results_csv = f"{runs_dir}/{latest_run}/results.csv"
if os.path.exists(results_csv):
    with open(results_csv) as f:
        lines = f.readlines()
    print(f"\nTotal epochs logged: {len(lines) - 1}")
    print("Header:", lines[0].strip())
    print("Last epoch row:", lines[-1].strip())
else:
    print("No results.csv found — training may not have completed or logging was interrupted.")

Weights folder contents: ['last.pt', 'best.pt', 'epoch0.pt', 'epoch10.pt', 'epoch20.pt', 'epoch30.pt', 'epoch40.pt', 'epoch50.pt']

Total epochs logged: 60
Header: epoch,time,train/box_loss,train/seg_loss,train/cls_loss,train/dfl_loss,train/sem_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),metrics/precision(M),metrics/recall(M),metrics/mAP50(M),metrics/mAP50-95(M),val/box_loss,val/seg_loss,val/cls_loss,val/dfl_loss,val/sem_loss,lr/pg0,lr/pg1,lr/pg2
Last epoch row: 60,4664.62,0.4072,1.03742,0.51763,1.0069,0,0.80685,0.60749,0.67847,0.59221,0.75481,0.58898,0.65601,0.56494,0.38879,0.71324,0.53526,0.93336,0,2.65e-05,2.65e-05,2.65e-05
